<details>
<summary>📊 YouTube Data Collection</summary>

# 📺 YouTube Data Collection
    ↓
## 1. Import Libraries
    ↓
## 2. API Configuration
    ↓
## 3. API Connection
    ↓
## 4. Channel Statistics
    ↓
## 5. Video Extraction
    ↓
## 6. Data Cleaning
    ↓
## 7. Feature Engineering
    ↓
## 8. Export Data
</details>

# 📺 YouTube Data Collection — Required Libraries

**Purpose:** Imported libraries required for collecting and preparing publicly available YouTube data through the YouTube Data API.

The libraries are grouped into the following categories:

- **YouTube API** → Used to connect with the YouTube Data API and fetch public channel, video, playlist, and statistics data.
- **Pandas** → Used to store and organize the collected API data into DataFrames for further processing.
- **Date & Time** → Used to parse YouTube publishing dates and handle video duration formats.
- **Jupyter Display** → Used to display raw API responses in a readable JSON format inside the notebook.

These libraries provide the basic foundation for the **YouTube public data collection pipeline** and prepare the data for further analysis and Power BI dashboard development.

In [1]:
from googleapiclient.discovery import build
from dateutil import parser
import pandas as pd
from IPython.display import JSON
from isodate import parse_time,parse_date,parse_datetime
import isodate

# 🔑 YouTube API Configuration

**Purpose:** Configure the YouTube Data API credentials required to collect publicly available YouTube channel and video data.

- **API Key** → Used to authenticate requests sent to the YouTube Data API.
- **Channel ID** → Identifies the specific YouTube channel from which public data will be collected.
- **Multiple Channel IDs** → Can be provided in a list to collect and compare data from multiple YouTube channels.

The API key should be kept **private** and should not be uploaded to GitHub or shared publicly.

In [ ]:
api_key="Use Your API Key"
channel_ids=["Use Your Channel IDs"]

# 🔌 YouTube API Connection

**Purpose:** Establish a connection with the YouTube Data API using the configured API key.

- **API Service Name** → Specifies that the YouTube API is being used.
- **API Version** → Defines the YouTube Data API version used for the project.
- **YouTube API Client** → Creates an authenticated API client using the API key.

This connection will be used to send requests to the YouTube API and collect publicly available channel and video data.

In [3]:
api_service_name = "youtube"
api_version = "v3"
youtube = build(api_service_name,api_version,developerKey=api_key)

# 📊 YouTube Data Extraction Functions

**Purpose:** Create reusable functions to collect channel-level, video-level, and video statistics data from the YouTube Data API.

### 🔹 GCS — Get Channel Statistics
Fetches important channel-level information such as:

- Channel Name
- Subscriber Count
- Total Views
- Total Videos
- Uploads Playlist ID

The function returns the collected channel information as a **Pandas DataFrame**.

### 🔹 GVI — Get Video IDs
Fetches all video IDs from a channel's uploads playlist.

- Retrieves up to 50 videos per API request.
- Uses `nextPageToken` to collect additional videos.
- Continues until all available videos are collected.

The function returns a **list of video IDs**.

### 🔹 GVD — Get Video Details
Fetches detailed information for all collected video IDs.

The function collects:

- Video ID
- Channel Name
- Title
- Description
- Tags
- Published Date
- Views
- Likes
- Comments
- Video Duration
- Video Definition
- Captions Availability

The API allows up to **50 video IDs per request**, so the function processes the IDs in batches.

The collected information is returned as a **Pandas DataFrame**.

### 🔄 Data Collection Flow

**Channel ID → Channel Statistics → Uploads Playlist → Video IDs → Video Details → DataFrame**

These functions create the core **YouTube public data extraction pipeline** for the project.

In [4]:
def GCS(youtube,channel_ids):
    all_data = []
    request = youtube.channels().list(part="snippet,contentDetails,statistics",id=','.join(channel_ids))
    response = request.execute()
    for item in response['items']:
        data = {'channelName': item['snippet']['title'],
                'subscribers': item['statistics']['subscriberCount'],
                'views': item['statistics']['viewCount'],
                'totalVideos': item['statistics']['videoCount'],
                'playlistId': item['contentDetails']['relatedPlaylists']['uploads']}
        all_data.append(data)  
    return pd.DataFrame(all_data)
def GVI(youtube, playlist_id): 
    video_ids = []
    request = youtube.playlistItems().list(part="snippet,contentDetails",playlistId=playlist_id,maxResults = 50)
    response = request.execute()
    for item in response['items']:
        video_ids.append(item['contentDetails']['videoId'])
    next_page_token = response.get('nextPageToken')
    while next_page_token is not None:
        request = youtube.playlistItems().list(part='contentDetails',playlistId = playlist_id,maxResults = 50,pageToken = next_page_token)
        response = request.execute()
        for item in response['items']:
            video_ids.append(item['contentDetails']['videoId'])
        next_page_token = response.get('nextPageToken')
    return video_ids
def GVD(youtube, video_ids):
    all_video_info = []
    for i in range(0, len(video_ids), 50):
        request = youtube.videos().list(part="snippet,contentDetails,statistics",id=','.join(video_ids[i:i+50]))
        response = request.execute() 
        for video in response['items']:
            stats_to_keep = {'snippet': ['channelTitle', 'title', 'description', 'tags', 'publishedAt'],
                             'statistics': ['viewCount', 'likeCount', 'favouriteCount', 'commentCount'],
                             'contentDetails': ['duration', 'definition', 'caption']}
            video_info = {}
            video_info['video_id'] = video['id']
            for k in stats_to_keep.keys():
                for v in stats_to_keep[k]:
                    try:
                        video_info[v] = video[k][v]
                    except:
                        video_info[v] = None
            all_video_info.append(video_info)
    return pd.DataFrame(all_video_info)

# 📊 Fetch Channel Statistics

**Purpose:** Execute the `GCS` function to collect statistics for the specified YouTube channel(s).

The collected data is stored in the **`channel_stats`** DataFrame.

This includes:

- Channel Name
- Subscribers
- Total Views
- Total Videos
- Uploads Playlist ID

The resulting DataFrame will be used for further **data analysis and Power BI dashboard development**.

In [5]:
channel_stats=GCS(youtube,channel_ids)

# 🎬 Extract Video IDs

**Purpose:** Retrieve the uploads playlist ID from the selected channel and use it to collect all available video IDs.

- **Playlist ID** → Identifies the channel's uploads playlist.
- **Video IDs** → The `GVI` function uses the playlist ID to retrieve the IDs of all videos uploaded by the channel.
- **Result** → The collected video IDs are stored in the `video_ids` list.

### 🔄 Flow

**Channel Statistics → Uploads Playlist ID → Video IDs**

These video IDs will be used in the next step to fetch detailed information and statistics for each video.

In [6]:
playlist_id=channel_stats['playlistId'][0]
video_ids=GVI(youtube, playlist_id)

# 🎥 Create Video DataFrame

**Purpose:** Execute the `GVD` function to collect detailed information and statistics for all the video IDs.

The extracted video data is stored in the **`df` DataFrame**.

The DataFrame contains information such as:

- Video ID
- Channel Name
- Video Title
- Description
- Tags
- Published Date
- Views
- Likes
- Comments
- Video Duration
- Video Definition
- Captions

### 🔄 Flow

**Video IDs → YouTube API → Video Details & Statistics → DataFrame (`df`)**

The `df` DataFrame is now ready for **data cleaning, analysis, and Power BI dashboard development**.

In [7]:
df=GVD(youtube,video_ids)

# 🔢 Convert Numeric Columns

**Purpose:** Convert YouTube engagement and performance columns into numeric data types for accurate analysis.

The following columns are converted:

- **View Count** → Total number of video views
- **Like Count** → Total number of likes
- **Favourite Count** → Total number of favourites
- **Comment Count** → Total number of comments

`errors='coerce'` converts invalid or missing values into **NaN** instead of causing an error.

This ensures the columns are ready for **calculations, statistical analysis, and Power BI visualizations**.

In [8]:
numeric_cols = ['viewCount', 'likeCount', 'favouriteCount', 'commentCount']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors = 'coerce', axis = 1).astype('Int32')

# 📅 Process Published Date

**Purpose:** Convert the YouTube `publishedAt` values into proper date-time objects and extract the day name for each video.

- **Published Date** → Converts the original YouTube date-time string into a Python date-time format.
- **Publish Day Name** → Extracts the day of the week, such as Monday, Tuesday, or Sunday.
- **Result** → Creates a new `pushblishDayName` column containing the day on which each video was published.

This helps analyze **which days of the week the channel publishes videos** and identify publishing patterns for further analysis and Power BI visualizations.

In [9]:
df['publishedAt'] = df['publishedAt'].apply(lambda x: parser.parse(x)) 
df['pushblishDayName'] = df['publishedAt'].apply(lambda x: x.strftime("%A"))

# ⏱️ Convert Video Duration

**Purpose:** Convert YouTube video duration from ISO 8601 format into seconds for easier analysis.

- **Duration** → YouTube stores video duration in ISO 8601 format, such as `PT10M32S`.
- **Parse Duration** → Converts the ISO 8601 duration into a Python time duration.
- **Duration in Seconds** → Converts the duration into total seconds.
- **Result** → Creates the `durationSecs` column containing the duration of each video in seconds.

This makes video duration easier to use for **calculations, comparisons, content analysis, and Power BI visualizations**.

In [10]:
df['durationSecs'] = df['duration'].apply(lambda x: isodate.parse_duration(x))
df['durationSecs'] = df['durationSecs'].astype('timedelta64[s]')
df['durationSecs'] = df['durationSecs'].dt.total_seconds().astype(int)

# 📅 Format Published Date

**Purpose:** Convert the `publishedAt` column into a proper date-time format and display it in a consistent format.

- **Date Conversion** → Converts the `publishedAt` values into Pandas date-time format.
- **Error Handling** → Invalid or missing date values are converted to `NaT` instead of causing an error.
- **Date Formatting** → Formats the date as `DD-MM-YYYY HH:MM:SS`.
- **Result** → Keeps the `publishedAt` column in a consistent and readable date-time format.

This makes the publishing date easier to use for **time-based analysis and Power BI visualizations**.

In [11]:
df["publishedAt"] = pd.to_datetime(df["publishedAt"],errors="coerce").dt.strftime("%d-%m-%Y %H:%M:%S")

# 📅 Extract Month from Published Date

**Purpose:** Extract the month from the formatted `publishedAt` column and convert it into a numeric data type.

- **Month Extraction** → Extracts the month value from the `DD-MM-YYYY` date format.
- **Numeric Conversion** → Converts the extracted month into `int32`.
- **Result** → Creates a `Month` column containing values from `1` to `12`.

This column can be used for **monthly upload trends, time-based analysis, and Power BI visualizations**.

In [12]:
df['Month'] = df['publishedAt'].str[3:5]
df['Month']=df['Month'].astype('int32')

# 📅 Extract Year from Published Date

**Purpose:** Extract the year from the formatted `publishedAt` column and convert it into a numeric data type.

- **Year Extraction** → Extracts the year from the `DD-MM-YYYY` date format.
- **Numeric Conversion** → Converts the extracted year into `int32`.
- **Result** → Creates a `Year` column containing the publication year.

This column can be used for **year-wise video analysis, upload trends, comparisons, and Power BI visualizations**.

In [13]:
df['Year'] = df['publishedAt'].str[6:10]
df['Year']=df['Year'].astype('int32')

# 🕐 Extract Publishing Hour

**Purpose:** Convert the `publishedAt` column back into a date-time format and extract the hour when each video was published.

- **Date-Time Conversion** → Converts `publishedAt` using the `DD-MM-YYYY HH:MM:SS` format.
- **Hour Extraction** → Extracts the hour from the publishing timestamp.
- **Result** → Creates an `hour` column containing values from `0` to `23`.

This helps analyze **which hours are most commonly used for publishing videos** and identify publishing patterns for Power BI analysis.

In [14]:
df['publishedAt'] = pd.to_datetime(df['publishedAt'],format='%d-%m-%Y %H:%M:%S')
df['hour'] = df['publishedAt'].dt.hour

# 🏷️ Calculate Tag Count

**Purpose:** Calculate the total number of tags associated with each YouTube video.

- **Tag Count** → Counts the number of tags available for each video.
- **Missing Tags** → Videos without tags are assigned a value of `0`.
- **Result** → Creates a `tagCount` column containing the total number of tags for each video.

This helps analyze **tag usage across videos** and identify relationships between the number of tags and video performance.

In [15]:
df['tagCount'] = df['tags'].apply(lambda x: 0 if x is None else len(x))

# 🗑️ Remove Unnecessary Column

**Purpose:** Remove the `favouriteCount` column from the DataFrame because it is not required for the analysis.

- **Column Removed** → `favouriteCount`
- **`inplace=True`** → Applies the change directly to the existing DataFrame.
- **Result** → The `favouriteCount` column is permanently removed from `df`.

This keeps the dataset **clean and focused on relevant YouTube performance metrics**.

In [16]:
df.drop(columns='favouriteCount' , inplace=True)

# 💾 Store YouTube Data in SQLite Database

**Purpose:** Save the processed YouTube video data and channel statistics into a local SQLite database for structured data storage and future analysis.

### 🔹 Data Preparation
The `tags` column is converted from a list format into a comma-separated text format so it can be stored properly in SQLite.

### 🔹 Database Connection
A connection is established with the **`youtube_data.db`** SQLite database.

### 🔹 Video Data
The video-level DataFrame is stored in a table named **`videos`**.

### 🔹 Channel Statistics
The channel-level DataFrame is stored in a table named **`channels`**.

### 🔹 Table Handling
Using `if_exists='replace'` ensures that existing tables with the same names are replaced with the latest dataset.

### 🔹 Close Connection
The database connection is closed after successfully storing the data.

### 🔄 Data Flow

**YouTube API → DataFrame → Data Preparation → SQLite Database → Videos & Channels Tables**

This creates a structured database that can later be connected to **SQL, Power BI, or other analytics tools**.

In [17]:
import sqlite3

In [18]:
df['tags'] = df['tags'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)
conn = sqlite3.connect('youtube_data.db')
df.to_sql('videos', conn, if_exists='replace', index=False)
channel_stats.to_sql('channels', conn, if_exists='replace', index=False)
conn.close()